In [ ]:
# ==========================================
# 1. SETUP & DATA PREPARATION 
# ==========================================
import re
import unicodedata
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.preprocessing import normalize
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from transformers import AutoTokenizer, AutoModel
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv
from torch_geometric.utils import coalesce  
from collections import Counter



device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
df = pd.read_csv('AFNC_news_dataset_tf-2.csv')

# --- Thai Normalization ---
ZW = ''.join(['\u200B', '\u200C', '\u200D', '\uFEFF'])
def normalize_thai(s):
    if pd.isna(s): return None
    s = str(s).replace('\u00A0', ' ').translate({ord(ch): None for ch in ZW})
    s = unicodedata.normalize('NFC', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s

df['ประเภทข่าว'] = df['ประเภทข่าว'].apply(normalize_thai)
df['หมวดหมู่ของข่าว'] = df['หมวดหมู่ของข่าว'].apply(normalize_thai)
df = df.dropna(subset=['หัวข้อข่าว', 'ประเภทข่าว']).reset_index(drop=True)

# Mapping
label2id = {c: i for i, c in enumerate(sorted(df['ประเภทข่าว'].unique()))}
id2label = {i: c for c, i in label2id.items()}
df['label_id'] = df['ประเภทข่าว'].map(label2id)

cat2id = {c: i for i, c in enumerate(sorted(df['หมวดหมู่ของข่าว'].unique()))}
id2cat = {i: c for c, i in cat2id.items()}
df['category_id'] = df['หมวดหมู่ของข่าว'].map(cat2id)

# ==========================================
# 2. BERT EMBEDDING (WangchanBERTa)
# ==========================================
model_WCB = "airesearch/wangchanberta-base-att-spm-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_WCB, use_fast=False)
lm_model = AutoModel.from_pretrained(model_WCB).to(device).eval()

@torch.no_grad()
def get_bert_embeddings(texts, batch_size=32):
    all_embs = []
    for i in range(0, len(texts), batch_size):
        batch = [str(t) for t in texts[i:i+batch_size]]
        inputs = tokenizer(batch, truncation=True, padding=True, max_length=256, return_tensors='pt').to(device)
        outputs = lm_model(**inputs)
        mask = inputs['attention_mask'].unsqueeze(-1)
        emb = (outputs.last_hidden_state * mask).sum(dim=1) / mask.sum(dim=1)
        all_embs.append(emb.cpu().numpy())
    return np.vstack(all_embs)

# สร้าง Embedding เริ่มต้น
content_emb = get_bert_embeddings(df['หัวข้อข่าว'].tolist())
x_np = normalize(content_emb, axis=1, norm='l2')



In [3]:
# ==========================================
# 3. BALANCING DATA 
# ==========================================
print("\n🔍 Checking Columns...")
print(f"ชื่อคอลัมน์ที่มีในเครื่องคุณ: {df.columns.tolist()}")

# พยายามหาคอลัมน์ที่มีคำว่า 
possible_date_cols = [c for c in df.columns if 'วัน' in c or 'เวลา' in c or 'date' in c.lower()]
if not possible_date_cols:
    print("⚠️ ไม่พบคอลัมน์วันที่! จะใช้วิธีสุ่มข้อมูลแทนการเรียงตามวันที่")
    date_col = None
else:
    date_col = possible_date_cols[0] # เลือกคอลัมน์แรกที่เจอ
    print(f"✅ ใช้คอลัมน์วันที่: '{date_col}'")

def convert_thai_date(date_str):
    if pd.isna(date_str): return pd.NaT
    s = str(date_str).strip().split(' ')[0] # เอาแค่ส่วนวันที่
    s = s.replace('/', '-') # ทำความสะอาดเบื้องต้น
    p = s.split('-')
    if len(p) == 3:
        # ถ้าปีเป็น พ.ศ. (เกิน 2400) ให้ลบ 543
        try:
            year = int(p[2])
            if year > 2400: p[2] = str(year - 543)
            # กรณีเจอวันที่แบบ DD-MM-YYYY
            return f"{p[2]}-{p[1]}-{p[0]}"
        except: return pd.NaT
    return pd.NaT

if date_col:
    df['parsed_date'] = pd.to_datetime(df[date_col].apply(convert_thai_date), errors='coerce')
    # ถ้าแปลงแล้วเป็น NaT หมดเลย ให้เลิกใช้คอลัมน์นี้
    if df['parsed_date'].isna().all():
        print("⚠️ ข้อมูลในคอลัมน์วันที่รูปแบบไม่ถูกต้อง ข้ามการเรียงลำดับ")
        sort_col = None
    else:
        sort_col = 'parsed_date'
else:
    sort_col = None

# --- ส่วนการทำ Balanced Dataset ---
idx_real = np.where(df['label_id'] == 0)[0]
idx_fake = np.where(df['label_id'] == 1)[0]
min_len = min(len(idx_real), len(idx_fake))

if sort_col:
    idx_real_bal = df.loc[idx_real].sort_values(sort_col, ascending=False).head(min_len).index
    idx_fake_bal = df.loc[idx_fake].sort_values(sort_col, ascending=False).head(min_len).index
else:
    idx_real_bal = idx_real[:min_len]
    idx_fake_bal = idx_fake[:min_len]

idx_balanced = np.concatenate([idx_real_bal, idx_fake_bal])
df_balanced = df.loc[idx_balanced].reset_index(drop=True)

# สำคัญ: ต้อง Slice x_np ให้ตรงกับ idx_balanced ด้วย
x_balanced = x_np[idx_balanced]
y_balanced = df_balanced['label_id'].values

print(f"✅ Balanced Dataset Ready: {len(df_balanced)} samples")


🔍 Checking Columns...
ชื่อคอลัมน์ที่มีในเครื่องคุณ: ['วันและเวลาที่เผยแพร่  ', 'ลิงค์ข่าว', 'หัวข้อข่าว', 'เนื้อหาข่าว', 'หน่วยงานที่ตรวจสอบ', 'ประเภทข่าว', 'หมวดหมู่ของข่าว', 'จำนวนเข้าชม', 'Hashtag', 'label_binary', 'label_id', 'category_id']
✅ ใช้คอลัมน์วันที่: 'วันและเวลาที่เผยแพร่  '
✅ Balanced Dataset Ready: 5744 samples


In [4]:
# ==========================================
# 4. GNN MODEL & GRAPH CONSTRUCTION
# ==========================================
class GCNNet(nn.Module):
    def __init__(self, num_node_features, num_classes, hidden_channels=256, dropout_rate=0.4):
        super().__init__()
        self.conv1 = GCNConv(num_node_features, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, num_classes)
        self.dropout_rate = dropout_rate

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        edge_weight = getattr(data, 'edge_attr', None)
        x = F.relu(self.conv1(x, edge_index, edge_weight=edge_weight))
        x = F.dropout(x, p=self.dropout_rate, training=self.training)
        x = self.conv2(x, edge_index, edge_weight=edge_weight)
        return x

# สร้าง kNN Graph
k = 10
nbrs = NearestNeighbors(n_neighbors=k, metric='cosine').fit(x_balanced)
dists, indices = nbrs.kneighbors(x_balanced)
src = np.repeat(np.arange(len(x_balanced)), k-1)
dst = indices[:, 1:].reshape(-1)
edge_index = torch.tensor(np.vstack([src, dst]), dtype=torch.long)
edge_weight = torch.tensor(1.0 - dists[:, 1:].reshape(-1), dtype=torch.float)

data = Data(x=torch.tensor(x_balanced, dtype=torch.float), 
            y=torch.tensor(y_balanced, dtype=torch.long),
            edge_index=edge_index, edge_attr=edge_weight).to(device)

model_gnn = GCNNet(768, 2).to(device)



In [5]:
# ==========================================
# 5. [DEBUG] TRACE & ERROR ANALYSIS FUNCTION
# ==========================================
def trace_and_analyze_error(target_idx, data, model, df_source):
    model.eval()
    with torch.no_grad():
        logits = model(data)
        probs = torch.softmax(logits, dim=1)
        pred = logits.argmax(dim=1)[target_idx].item()
        actual = data.y[target_idx].item()
        
    print("\n" + "🔍" * 15)
    print(f"ERROR ANALYSIS & TRACE: NODE {target_idx}")
    print(f"Headline: {df_source.iloc[target_idx]['หัวข้อข่าว']}")
    print(f"Result: Actual={id2label[actual]} | Pred={id2label[pred]} | Conf={probs[target_idx][pred]:.4f}")
    print("🔍" * 15)

    # --- Step 1: BERT Word Importance ---
    inputs = tokenizer(df_source.iloc[target_idx]['หัวข้อข่าว'], return_tensors='pt', truncation=True).to(device)
    with torch.no_grad():
        out = lm_model(**inputs, output_attentions=True)
    # ดูความสำคัญของคำจาก Attention Head สุดท้าย
    attn = out.attentions[-1][0].mean(dim=0).mean(dim=0).cpu().numpy()
    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
    top_words = sorted(zip(tokens, attn), key=lambda x: x[1], reverse=True)[:5]
    print(f"\n[1] Word Importance (BERT):")
    for t, s in top_words: 
        if t not in ['<s>', '</s>', '<pad>']: print(f"   - {t:15} score: {s:.4f}")

    # --- Step 2: GNN Influence (Neighbors) ---
    row, col = data.edge_index
    neighbors = col[row == target_idx]
    neighbor_labels = [id2label[data.y[n].item()] for n in neighbors]
    neighbor_counts = Counter(neighbor_labels)
    
    print(f"\n[2] Neighbor Influence (Graph):")
    print(f"   - Total Neighbors: {len(neighbors)}")
    print(f"   - Label Distribution in Neighbors: {dict(neighbor_counts)}")
    
    if neighbor_counts[id2label[pred]] > neighbor_counts[id2label[actual]]:
        print("   🚩 Root Cause: 'Peer Pressure' - โดนกลุ่มเพื่อนดึงคะแนนไปทางคลาสที่ผิด")
    else:
        print("   🚩 Root Cause: 'Feature Ambiguity' - เนื้อหาข่าวเองมีความก้ำกึ่งสูง")

In [6]:
# ==========================================
# 6. RUN DEBUGGING
# ==========================================
# พยายามโหลด Weights (ถ้ามี)
try:
    model_gnn.load_state_dict(torch.load('best_model.pth', map_location=device))
    print("\n✅ Loaded best_model.pth")
except:
    print("\n⚠️ Weights not found, tracing with initial state.")

# เลือก Node แรกมาลอง Trace
trace_and_analyze_error(0, data, model_gnn, df_balanced)

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
CamembertSdpaSelfAttention is used but `torch.nn.functional.scaled_dot_product_attention` does not support non-absolute `position_embedding_type` or `output_attentions=True` or `head_mask`. Falling back to the manual attention implementation, but specifying the manual implementation will be required from Transformers version v5.0.0 onwards. This warning can be removed using the argument `attn_implementation="eager"` when loading the model.



✅ Loaded best_model.pth

🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍
ERROR ANALYSIS & TRACE: NODE 0
Headline: ไทยเรียกร้องให้กัมพูชารับผิดชอบ กรณียิงข้ามแดนเข้ามาฝั่งไทย จนทำให้กำลังพลได้รับบาดเจ็บ 1 นาย
Result: Actual=ข่าวจริง | Pred=ข่าวจริง | Conf=0.5299
🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍

[1] Word Importance (BERT):
   - รับผิดชอบ       score: 0.2254
   - เรียกร้อง       score: 0.2254
   - กัมพูชา         score: 0.0204

[2] Neighbor Influence (Graph):
   - Total Neighbors: 9
   - Label Distribution in Neighbors: {'ข่าวปลอม': 5, 'ข่าวจริง': 4}
   🚩 Root Cause: 'Feature Ambiguity' - เนื้อหาข่าวเองมีความก้ำกึ่งสูง


In [7]:
import torch
import numpy as np

def trace_with_formulas(node_idx, data, model, df_source):
    model.eval()
    headline = df_source.iloc[node_idx]['หัวข้อข่าว']
    
    print("\n" + "🔬" * 10 + " GNN DECISION LOGIC " + "🔬" * 10)
    print(f"HEADLINE: {headline}")
    print("=" * 60)

    with torch.no_grad():
        # --- ขั้นที่ 1: BERT (The Semantic Gauge) ---
        x_bert = data.x[node_idx]
        print(f"\n[STEP 1] BERT Embedding: 'การตีความเนื้อหา'")
        print(f"   📝 สูตร: Text -> Vector X (768 dimensions)")
        print(f"   💡 เกณฑ์: BERT จะมองหา 'คำสำคัญ' (Keywords) และ 'บริบท' (Context)")
        print(f"      ถ้าเจอคำที่มักอยู่ในข่าวปลอม เช่น 'ด่วนที่สุด', 'หลุดออกมา', 'สรรพคุณมหัศจรรย์'")
        print(f"      ตัวเลขใน Vector X จะขยับไปอยู่ในโซนของ 'ความไม่น่าเชื่อถือ'")

        # --- ขั้นที่ 2: Graph Influence (The Peer Review) ---
        row, col = data.edge_index
        neighbors = col[row == node_idx]
        neighbor_labels = [id2label[data.y[n].item()] for n in neighbors]
        
        print(f"\n[STEP 2] Graph Connectivity: 'กฎการรวมกลุ่ม'")
        print(f"   🤝 สูตร: kNN (Cosine Similarity) -> หาโหนดที่ใกล้กันที่สุด {len(neighbors)} อัน")
        print(f"   💡 เกณฑ์: 'คุณคือค่าเฉลี่ยของเพื่อนที่คุณคบ' (Homophily Property)")
        print(f"      โมเดลเชื่อว่าข่าวประเภทเดียวกันมักจะอ้างถึงแหล่งข่าวคล้ายกันหรือเขียนคล้ายกัน")
        print(f"      ในเคสนี้ เพื่อนบ้านของคุณคือ: {dict(Counter(neighbor_labels))}")

        # --- ขั้นที่ 3: GCN Layer 1 (Message Passing) ---
        w1 = model.conv1.lin.weight
        z1_me = torch.matmul(x_bert, w1.T) + model.conv1.bias
        
        neighbor_features = data.x[neighbors]
        z1_neighbors = torch.matmul(neighbor_features, w1.T) + model.conv1.bias
        z1_agg = torch.mean(z1_neighbors, dim=0) 
        
        print(f"\n[STEP 3] GCN Aggregation: 'การแลกเปลี่ยนข้อมูล'")
        print(f"   🌀 สูตร: h_v = ReLU( W * mean(h_u for u in neighbors) )")
        print(f"   💡 เกณฑ์: เป็นการ 'ถ่วงดุล' ระหว่าง สิ่งที่ตัวเองเป็น (BERT) กับ สิ่งที่สังคมรอบข้างบอก (Neighbors)")
        print(f"      ตัวเลข {z1_me[:3].cpu().numpy().round(3)} (จากตัวเอง) จะถูกนำไปผสมกับ")
        print(f"      ตัวเลข {z1_agg[:3].cpu().numpy().round(3)} (จากคนอื่น) เพื่อหาจุดกึ่งกลางของความจริง")

        # --- ขั้นที่ 4: Final Squeeze & Prediction ---
        logits = model(data)[node_idx]
        probs = torch.softmax(logits, dim=0)
        
        print(f"\n[STEP 4] Decision Output: 'การตัดสินสุดท้าย'")
        print(f"   ⚖️ สูตร: Softmax(z) = exp(z_i) / sum(exp(z_j))")
        print(f"   💡 เกณฑ์ตัดสิน: ใครได้แต้มมากกว่า (Score 0 vs Score 1) คือผู้ชนะ")
        print(f"      แต้มข่าวจริง: {probs[0]:.4f} | แต้มข่าวปลอม: {probs[1]:.4f}")

        print("\n" + "📊" * 5 + " สรุปเกณฑ์ที่โมเดลใช้ " + "📊" * 5)
        # วิเคราะห์ผลต่าง
        if neighbor_labels.count(id2label[data.y[node_idx].item()]) < len(neighbors)/2:
            print("🚩 ข่าวนี้ทายยาก: เพราะเนื้อหาข้างในดูเป็น 'ข่าวจริง' แต่คนรอบข้างส่วนใหญ่เป็น 'ข่าวปลอม'")
        else:
            print("✅ ข่าวนี้ทายง่าย: ทั้งเนื้อหาและกลุ่มเพื่อนไปในทิศทางเดียวกัน")

trace_with_formulas(0, data, model_gnn, df_balanced)


🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬 GNN DECISION LOGIC 🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬
HEADLINE: ไทยเรียกร้องให้กัมพูชารับผิดชอบ กรณียิงข้ามแดนเข้ามาฝั่งไทย จนทำให้กำลังพลได้รับบาดเจ็บ 1 นาย

[STEP 1] BERT Embedding: 'การตีความเนื้อหา'
   📝 สูตร: Text -> Vector X (768 dimensions)
   💡 เกณฑ์: BERT จะมองหา 'คำสำคัญ' (Keywords) และ 'บริบท' (Context)
      ถ้าเจอคำที่มักอยู่ในข่าวปลอม เช่น 'ด่วนที่สุด', 'หลุดออกมา', 'สรรพคุณมหัศจรรย์'
      ตัวเลขใน Vector X จะขยับไปอยู่ในโซนของ 'ความไม่น่าเชื่อถือ'

[STEP 2] Graph Connectivity: 'กฎการรวมกลุ่ม'
   🤝 สูตร: kNN (Cosine Similarity) -> หาโหนดที่ใกล้กันที่สุด 9 อัน
   💡 เกณฑ์: 'คุณคือค่าเฉลี่ยของเพื่อนที่คุณคบ' (Homophily Property)
      โมเดลเชื่อว่าข่าวประเภทเดียวกันมักจะอ้างถึงแหล่งข่าวคล้ายกันหรือเขียนคล้ายกัน
      ในเคสนี้ เพื่อนบ้านของคุณคือ: {'ข่าวปลอม': 5, 'ข่าวจริง': 4}

[STEP 3] GCN Aggregation: 'การแลกเปลี่ยนข้อมูล'
   🌀 สูตร: h_v = ReLU( W * mean(h_u for u in neighbors) )
   💡 เกณฑ์: เป็นการ 'ถ่วงดุล' ระหว่าง สิ่งที่ตัวเองเป็น (BERT) กับ สิ่งที่สังคมรอบข้างบอก (Neighbors)
   

In [8]:
import torch
import numpy as np

def deep_dive_trace(target_idx, data, model):
    model.eval()
    print(f"\n{'='*20} 🔬 DEEP DIVE CALCULATION: Node {target_idx} {'='*20}")
    
    with torch.no_grad():
        # ---------------------------------------------------------
        # 1. INPUT STAGE: ข้อมูลดิบ (Feature Vector)
        # ---------------------------------------------------------
        # ดึง Feature ของ Node ที่เราสนใจออกมา
        x_input = data.x[target_idx] 
        print(f"\n[STEP 1] Input Features (x) ขนาด {x_input.shape}")
        print(f"   ค่าดิบ (5 ตัวแรก): {x_input[:5].cpu().numpy().round(4)}")
        print("   คำอธิบาย: นี่คือตัวเลขที่ได้จาก BERT (Embeddings) เป็นตัวแทนความหมายของประโยค")

        # ---------------------------------------------------------
        # 2. LAYER 1: Linear Transformation (W * x + b)
        # ---------------------------------------------------------
        # ดึง Weight และ Bias จาก Layer แรก (conv1)
        # GCNConv พื้นฐานทำงานคือ (AxW) แต่ใน TorchGeometric มักทำ Linear ก่อนส่ง Message
        weight1 = model.conv1.lin.weight  # shape [out_dim, in_dim]
        bias1 = model.conv1.bias          # shape [out_dim]
        
        # จำลองการคูณ Matrix: Linear = (x @ W.T) + b
        # นี่คือการ "แปรรูปความหมาย" จาก 768 มิติ -> 256 มิติ
        linear_trans = torch.matmul(x_input, weight1.T) + bias1
        
        print(f"\n[STEP 2] Linear Transformation (W @ x + b)")
        print(f"   Weight Shape: {weight1.shape}")
        print(f"   ผลลัพธ์การคูณ (5 ตัวแรก): {linear_trans[:5].cpu().numpy().round(4)}")
        print("   คำอธิบาย: โมเดลกำลังผสม 'คำ' ต่างๆ เข้าด้วยกันผ่าน Weight ที่เรียนรู้มา")

        # ---------------------------------------------------------
        # 3. MESSAGE PASSING: การรับข้อมูลจากเพื่อนบ้าน
        # ---------------------------------------------------------
        row, col = data.edge_index
        neighbor_indices = col[row == target_idx] # หาว่าใครเป็นเพื่อนเราบ้าง
        
        print(f"\n[STEP 3] Neighbor Aggregation (Message Passing)")
        print(f"   จำนวนเพื่อนบ้าน: {len(neighbor_indices)} nodes")
        
        # คำนวณ Linear Transform ของเพื่อนแต่ละคน เพื่อเตรียมส่งมาให้เรา
        # สูตร GCN ปกติ: h_v = Sum( c_ij * W * x_j )
        neighbors_features = data.x[neighbor_indices]
        neighbors_transformed = torch.matmul(neighbors_features, weight1.T) + bias1
        
        # โชว์ตัวอย่างเพื่อนคนแรก
        if len(neighbor_indices) > 0:
            print(f"   ตัวอย่างเพื่อนคนแรก (Node {neighbor_indices[0].item()}) ส่งค่ามา: {neighbors_transformed[0][:5].cpu().numpy().round(4)}")
        
        # ทำการรวมค่า (Aggregation) - GCNConv ใช้ค่า Mean (หรือ Weighted Sum ตาม norm)
        # เพื่อความง่ายในการดู เราจะใช้ Mean จำลอง (GCN ของจริงจะมี normalization factor)
        agg_signal = torch.mean(neighbors_transformed, dim=0)
        print(f"   ค่าเฉลี่ยที่ได้รับจากเพื่อน (Aggregated): {agg_signal[:5].cpu().numpy().round(4)}")

        # รวมร่าง: ตัวเรา + ข้อมูลจากเพื่อน (implementation ของ GCNConv จะรวม topology ในขั้นตอนนี้)
        # ใน GCNConv แบบพื้นฐานสุด (Kipf) ข้อมูลจะผ่าน Adjacency Matrix แล้ว
        # สมมติว่าเป็นผลรวมของการคำนวณจริงจาก Library
        out_conv1 = model.conv1(data.x, data.edge_index)[target_idx]
        
        print(f"   Result after Layer 1 (Actual from Model): {out_conv1[:5].cpu().numpy().round(4)}")

        # ---------------------------------------------------------
        # 4. ACTIVATION & DROPOUT
        # ---------------------------------------------------------
        # ReLU: ตัดค่าลบให้เป็น 0
        relu_out = torch.relu(out_conv1)
        print(f"\n[STEP 4] ReLU Activation (ตัดค่าลบ)")
        print(f"   Before ReLU: {out_conv1[:5].cpu().numpy().round(4)}")
        print(f"   After ReLU : {relu_out[:5].cpu().numpy().round(4)}")
        
        # Dropout (เฉพาะตอน Train, ตอนนี้ Eval จะไม่ทำงาน แต่โชว์ให้เห็น flow)
        # ถ้า Training: บางค่าจะถูกสุ่มให้เป็น 0

        # ---------------------------------------------------------
        # 5. FINAL CLASSIFICATION (Layer 2)
        # ---------------------------------------------------------
        # ยุบจาก 256 มิติ -> 2 มิติ (Real, Fake)
        weight2 = model.conv2.lin.weight
        bias2 = model.conv2.bias
        
        # Final Logits = (Hidden @ W2.T) + b2 (รวม Aggregation จากเพื่อนอีกรอบใน Layer 2)
        # เราดูผลลัพธ์สุดท้ายจากโมเดลเลย
        logits = model(data)[target_idx]
        
        print(f"\n[STEP 5] Final Output (Logits)")
        print(f"   สูตร: y = W2 * h_hidden + b2")
        print(f"   ค่าคะแนนดิบ (Logits): {logits.cpu().numpy().round(4)}")
        print(f"      [Class 0 - ข่าวจริง]: {logits[0]:.4f}")
        print(f"      [Class 1 - ข่าวปลอม]: {logits[1]:.4f}")

        # ---------------------------------------------------------
        # 6. SOFTMAX PROBABILITY
        # ---------------------------------------------------------
        probs = torch.softmax(logits, dim=0)
        print(f"\n[STEP 6] Probability (Softmax)")
        print(f"   สูตร: e^x_i / sum(e^x)")
        print(f"   ความน่าจะเป็น: {probs.cpu().numpy().round(4)}")
        
        pred_label = logits.argmax().item()
        actual_label = data.y[target_idx].item()
        print(f"\n🎯 สรุปผล: ทำนาย {id2label[pred_label]} ({probs[pred_label]*100:.2f}%) | เฉลย {id2label[actual_label]}")

# เรียกใช้งานฟังก์ชัน
# ลองเปลี่ยนเลข 0 เป็นเลข index อื่นๆ เพื่อดูข่าวอื่น
deep_dive_trace(0, data, model_gnn)


==================== 🔬 DEEP DIVE CALCULATION: Node 0 ====================

[STEP 1] Input Features (x) ขนาด torch.Size([768])
   ค่าดิบ (5 ตัวแรก): [ 0.03   -0.0434  0.0243  0.0146 -0.034 ]
   คำอธิบาย: นี่คือตัวเลขที่ได้จาก BERT (Embeddings) เป็นตัวแทนความหมายของประโยค

[STEP 2] Linear Transformation (W @ x + b)
   Weight Shape: torch.Size([256, 768])
   ผลลัพธ์การคูณ (5 ตัวแรก): [0.0616 0.0527 0.0428 0.0511 0.0437]
   คำอธิบาย: โมเดลกำลังผสม 'คำ' ต่างๆ เข้าด้วยกันผ่าน Weight ที่เรียนรู้มา

[STEP 3] Neighbor Aggregation (Message Passing)
   จำนวนเพื่อนบ้าน: 9 nodes
   ตัวอย่างเพื่อนคนแรก (Node 3419) ส่งค่ามา: [0.0512 0.0657 0.0674 0.0561 0.0356]
   ค่าเฉลี่ยที่ได้รับจากเพื่อน (Aggregated): [0.0672 0.0548 0.0576 0.0591 0.0602]
   Result after Layer 1 (Actual from Model): [0.0778 0.0681 0.0747 0.0737 0.0644]

[STEP 4] ReLU Activation (ตัดค่าลบ)
   Before ReLU: [0.0778 0.0681 0.0747 0.0737 0.0644]
   After ReLU : [0.0778 0.0681 0.0747 0.0737 0.0644]

[STEP 5] Final Output (Logits)
   สู

In [9]:
def atomic_trace(target_idx, data, model):
    print(f"\n{'='*20} ⚛️ ATOMIC LEVEL TRACE: กำเนิดตัวเลข 1 ตัว {'='*20}")
    
    with torch.no_grad():
        # 1. เตรียมวัตถุดิบ
        x_input = data.x[target_idx]                 # Input 768 ตัว
        weight_row0 = model.conv1.lin.weight[0]      # Weight แถวที่ 0 (สำหรับสร้าง Output ตัวแรก)
        bias_val = model.conv1.bias[0].item()        # Bias ตัวแรก
        
        print(f"เราจะมาดูที่มาของตัวเลขแรกใน Layer 1 (ค่าเป้าหมายคือประมาณ {0.0616})")
        print(f"สูตรคือ: (Input[0]*Weight[0]) + (Input[1]*Weight[1]) + ... + Bias")
        print("-" * 60)

        # 2. จำลองการคูณทีละตัว (Dot Product Calculation)
        running_sum = 0
        
        # แสดง 5 เทอมแรก
        print("🧮 การคูณและบวกกัน 768 ครั้ง (แสดง 5 ครั้งแรก):")
        for i in range(5):
            val_x = x_input[i].item()
            val_w = weight_row0[i].item()
            product = val_x * val_w
            running_sum += product
            
            print(f"   Term {i+1}: Input({val_x:.4f})  x  Weight({val_w:.4f})  =  {product:.6f}")
        
        # คำนวณส่วนที่เหลือแบบรวบรัด
        remaining_sum = torch.dot(x_input[5:], weight_row0[5:]).item()
        total_dot_product = running_sum + remaining_sum
        
        print(f"   ... (บวกอีก {768-5} คู่ที่เหลือ) ...")
        print(f"   ผลรวมส่วนที่เหลือ: {remaining_sum:.6f}")
        print("-" * 60)
        
        # 3. สรุปผลลัพธ์
        print(f"1️⃣ ผลรวม Dot Product (Sum): {total_dot_product:.6f}")
        print(f"2️⃣ บวกค่า Bias ({bias_val:.6f})")
        final_val = total_dot_product + bias_val
        
        print(f"✅ ผลลัพธ์สุดท้าย (Linear Result): {final_val:.6f}")
        
        # ตรวจสอบกับค่าจริงจากโมเดล
        actual_val = (torch.matmul(x_input, model.conv1.lin.weight.T) + model.conv1.bias)[0].item()
        print(f"🔎 เทียบกับค่าจริงจาก PyTorch:   {actual_val:.6f}")

atomic_trace(0, data, model_gnn)


==================== ⚛️ ATOMIC LEVEL TRACE: กำเนิดตัวเลข 1 ตัว ====================
เราจะมาดูที่มาของตัวเลขแรกใน Layer 1 (ค่าเป้าหมายคือประมาณ 0.0616)
สูตรคือ: (Input[0]*Weight[0]) + (Input[1]*Weight[1]) + ... + Bias
------------------------------------------------------------
🧮 การคูณและบวกกัน 768 ครั้ง (แสดง 5 ครั้งแรก):
   Term 1: Input(0.0300)  x  Weight(-0.0197)  =  -0.000590
   Term 2: Input(-0.0434)  x  Weight(-0.0836)  =  0.003632
   Term 3: Input(0.0243)  x  Weight(-0.0241)  =  -0.000585
   Term 4: Input(0.0146)  x  Weight(-0.0158)  =  -0.000231
   Term 5: Input(-0.0340)  x  Weight(0.0156)  =  -0.000530
   ... (บวกอีก 763 คู่ที่เหลือ) ...
   ผลรวมส่วนที่เหลือ: 0.025334
------------------------------------------------------------
1️⃣ ผลรวม Dot Product (Sum): 0.027030
2️⃣ บวกค่า Bias (0.034583)
✅ ผลลัพธ์สุดท้าย (Linear Result): 0.061613
🔎 เทียบกับค่าจริงจาก PyTorch:   0.061613


In [10]:
import torch
import numpy as np

def pipeline_inspector(node_idx, data, model, df_source):
    model.eval()
    print(f"\n{'='*60}")
    print(f"🚀 PIPELINE INSPECTION: Node {node_idx}")
    print(f"📰 Headline: {df_source.iloc[node_idx]['หัวข้อข่าว']}")
    print(f"{'='*60}\n")
    
    with torch.no_grad():
        # =========================================================
        # STEP 1: EMBEDDING (INPUT)
        # =========================================================
        # สิ่งที่เข้า: ข้อความข่าว (แปลงเป็น ID แล้ว)
        # สิ่งที่ออก: Vector 768 ช่อง
        x_in = data.x[node_idx]
        print(f"📍 STEP 1: Embedding (Input Layer)")
        print(f"   📥 Input:  ข้อความดิบ")
        print(f"   ⚙️ Process: BERT Embedding")
        print(f"   📤 Output: Vector ขนาด {list(x_in.shape)}")
        print(f"   📊 ตัวเลข 5 ตัวแรก: {x_in[:5].cpu().numpy().round(4)}")
        print(f"   ⬇️ ส่งต่อไปยัง Step 2\n")

        # =========================================================
        # STEP 2: LINEAR TRANSFORM (LAYER 1)
        # =========================================================
        # สิ่งที่เข้า: Vector 768 ช่อง
        # สิ่งที่ออก: Vector 256 ช่อง (เฉพาะส่วนของตัวเอง)
        w1 = model.conv1.lin.weight  # [256, 768]
        b1 = model.conv1.bias        # [256]
        
        # คำนวณ: (x * W^T) + b
        self_feat = torch.matmul(x_in, w1.T) + b1
        
        print(f"📍 STEP 2: Linear Transformation (Self-Feature)")
        print(f"   📥 Input:  Vector 768 ช่อง (จาก Step 1)")
        print(f"   ⚙️ Process: Matrix Multiply (x @ W.T + b)")
        print(f"   📤 Output: Vector ขนาด {list(self_feat.shape)}")
        print(f"   📊 ตัวเลข 5 ตัวแรก: {self_feat[:5].cpu().numpy().round(4)}")
        print(f"   ⬇️ ส่งต่อไปยัง Step 3\n")

        # =========================================================
        # STEP 3: NEIGHBOR AGGREGATION
        # =========================================================
        # สิ่งที่เข้า: Vector 256 ของเพื่อนๆ
        # สิ่งที่ออก: Vector 256 ที่รวมพลังกันแล้ว
        row, col = data.edge_index
        neighbor_ids = col[row == node_idx]
        
        # ดึง Feature ของเพื่อน -> แปลง Linear -> หาค่าเฉลี่ย
        # (หมายเหตุ: GCNConv ของจริงจะรวม Self-loop ด้วย แต่นี่จำลองให้เห็นภาพแยกกัน)
        neighbor_x = data.x[neighbor_ids]
        neighbor_trans = torch.matmul(neighbor_x, w1.T) + b1
        agg_feat = torch.mean(neighbor_trans, dim=0)
        
        # จำลองการรวม (GCN Aggregation)
        # ใน PyG GCNConv: output = (Self + Neighbors) normalized
        # เราจะดึงค่าจริงจากโมเดลมาเลยเพื่อความแม่นยำที่สุด
        gcn_output = model.conv1(data.x, data.edge_index)[node_idx]
        
        print(f"📍 STEP 3: Neighbor Aggregation (Message Passing)")
        print(f"   📥 Input:  เพื่อนบ้าน {len(neighbor_ids)} คน")
        print(f"   ⚙️ Process: Average & Normalize")
        print(f"   📤 Output: Vector ขนาด {list(gcn_output.shape)}")
        print(f"   📊 ตัวเลข 5 ตัวแรก: {gcn_output[:5].cpu().numpy().round(4)}")
        print(f"   ⬇️ ส่งต่อไปยัง Step 4\n")

        # =========================================================
        # STEP 4: ACTIVATION (RELU)
        # =========================================================
        # สิ่งที่เข้า: Vector 256 (ที่มีลบปนบวก)
        # สิ่งที่ออก: Vector 256 (ที่ไม่มีเลขลบ)
        relu_out = torch.relu(gcn_output)
        
        print(f"📍 STEP 4: Activation (ReLU)")
        print(f"   📥 Input:  Vector 256 ช่อง (จาก Step 3)")
        print(f"   ⚙️ Process: Cut Negative Values (x < 0 becomes 0)")
        print(f"   📤 Output: Vector ขนาด {list(relu_out.shape)}")
        print(f"   📊 ตัวเลข 5 ตัวแรก: {relu_out[:5].cpu().numpy().round(4)}")
        print(f"   ⬇️ ส่งต่อไปยัง Step 5\n")

        # =========================================================
        # STEP 5: CLASSIFIER (LOGITS)
        # =========================================================
        # สิ่งที่เข้า: Vector 256 (Feature ที่พร้อมใช้)
        # สิ่งที่ออก: Vector 2 (คะแนนดิบ 2 คลาส)
        w2 = model.conv2.lin.weight # [2, 256]
        b2 = model.conv2.bias       # [2]
        
        # เนื่องจาก GCNConv ชั้น 2 ก็มีการรวมเพื่อนอีกรอบ
        # เราจะดึงค่าจากโมเดลโดยตรงเพื่อความถูกต้อง
        final_logits = model(data)[node_idx]
        
        print(f"📍 STEP 5: Classification Layer")
        print(f"   📥 Input:  Vector 256 ช่อง (จาก Step 4 + เพื่อนรอบ 2)")
        print(f"   ⚙️ Process: Linear Projection to 2 Classes")
        print(f"   📤 Output: Vector ขนาด {list(final_logits.shape)}")
        print(f"   📊 คะแนนดิบ (Logits): {final_logits.cpu().numpy().round(4)}")
        print(f"   ⬇️ ส่งต่อไปยัง Step 6\n")

        # =========================================================
        # STEP 6: PROBABILITY (SOFTMAX)
        # =========================================================
        # สิ่งที่เข้า: คะแนนดิบ
        # สิ่งที่ออก: % ความน่าจะเป็น
        probs = torch.softmax(final_logits, dim=0)
        
        print(f"📍 STEP 6: Final Decision (Softmax)")
        print(f"   📥 Input:  Logits {final_logits.cpu().numpy().round(4)}")
        print(f"   ⚙️ Process: e^x / sum(e^x)")
        print(f"   📤 Output: % Confidence")
        print(f"   📊 Result: [Real: {probs[0]:.4f}, Fake: {probs[1]:.4f}]")
        
        pred = probs.argmax().item()
        print(f"\n🏁 FINAL ANSWER: {id2label[pred]} (มั่นใจ {probs[pred]*100:.2f}%)")

# เรียกใช้งาน (ลองเปลี่ยนเลข 0 เป็นเลขอื่นเพื่อดูข่าวอื่น)
pipeline_inspector(0, data, model_gnn, df_balanced)


🚀 PIPELINE INSPECTION: Node 0
📰 Headline: ไทยเรียกร้องให้กัมพูชารับผิดชอบ กรณียิงข้ามแดนเข้ามาฝั่งไทย จนทำให้กำลังพลได้รับบาดเจ็บ 1 นาย

📍 STEP 1: Embedding (Input Layer)
   📥 Input:  ข้อความดิบ
   ⚙️ Process: BERT Embedding
   📤 Output: Vector ขนาด [768]
   📊 ตัวเลข 5 ตัวแรก: [ 0.03   -0.0434  0.0243  0.0146 -0.034 ]
   ⬇️ ส่งต่อไปยัง Step 2

📍 STEP 2: Linear Transformation (Self-Feature)
   📥 Input:  Vector 768 ช่อง (จาก Step 1)
   ⚙️ Process: Matrix Multiply (x @ W.T + b)
   📤 Output: Vector ขนาด [256]
   📊 ตัวเลข 5 ตัวแรก: [0.0616 0.0527 0.0428 0.0511 0.0437]
   ⬇️ ส่งต่อไปยัง Step 3

📍 STEP 3: Neighbor Aggregation (Message Passing)
   📥 Input:  เพื่อนบ้าน 9 คน
   ⚙️ Process: Average & Normalize
   📤 Output: Vector ขนาด [256]
   📊 ตัวเลข 5 ตัวแรก: [0.0778 0.0681 0.0747 0.0737 0.0644]
   ⬇️ ส่งต่อไปยัง Step 4

📍 STEP 4: Activation (ReLU)
   📥 Input:  Vector 256 ช่อง (จาก Step 3)
   ⚙️ Process: Cut Negative Values (x < 0 becomes 0)
   📤 Output: Vector ขนาด [256]
   📊 ตัวเลข 5 ตัวแรก

In [11]:
import torch
import pandas as pd

def explain_decision_logic(target_idx, data, model):
    model.eval()
    with torch.no_grad():
        # 1. ดึง Feature สุดท้ายก่อนเข้าตัดสิน (Output ของ Step 4)
        # จำลองการรันผ่าน Layer 1 และ ReLU
        out_conv1 = model.conv1(data.x, data.edge_index)
        feature_vector = torch.relu(out_conv1)[target_idx] # ขนาด 256
        
        # 2. ดึง Weight ของชั้นตัดสิน (Step 5)
        # W shape: [2, 256] -> แถว 0 คือตัวคูณข่าวจริง, แถว 1 คือตัวคูณข่าวปลอม
        W_classifier = model.conv2.lin.weight
        b_classifier = model.conv2.bias
        
        # 3. คำนวณคะแนนแยกราย Feature (Element-wise multiplication)
        # ดูว่า Feature ไหนส่งผลต่อคะแนนข่าวจริง/ปลอมบ้าง
        score_real_contributions = feature_vector * W_classifier[0]
        score_fake_contributions = feature_vector * W_classifier[1]
        
        sum_real = score_real_contributions.sum() + b_classifier[0]
        sum_fake = score_fake_contributions.sum() + b_classifier[1]
        
        print(f"\n{'='*20} 🧠 LOGIC EXPLAINER: Node {target_idx} {'='*20}")
        print(f"Final Real Score: {sum_real:.4f}")
        print(f"Final Fake Score: {sum_fake:.4f}")
        
        # หา Top Features ที่ทำให้โมเดลคิดว่าเป็นข่าวจริง
        top_real_idx = torch.argsort(score_real_contributions, descending=True)[:3]
        print(f"\n✅ ปัจจัยบวก (ดันให้เป็นข่าวจริง):")
        for idx in top_real_idx:
            val = feature_vector[idx].item()
            weight = W_classifier[0][idx].item()
            score = score_real_contributions[idx].item()
            if val > 0: # โชว์เฉพาะที่มีค่า
                print(f"   - Feature[{idx}]: มีค่า {val:.2f} x Weight({weight:.2f}) = +{score:.4f} คะแนน")

        # หา Top Features ที่ทำให้โมเดลคิดว่าเป็นข่าวปลอม
        top_fake_idx = torch.argsort(score_fake_contributions, descending=True)[:3]
        print(f"\n❌ ปัจจัยลบ (ดึงให้เป็นข่าวปลอม):")
        for idx in top_fake_idx:
            val = feature_vector[idx].item()
            weight = W_classifier[1][idx].item()
            score = score_fake_contributions[idx].item()
            if val > 0:
                print(f"   - Feature[{idx}]: มีค่า {val:.2f} x Weight({weight:.2f}) = +{score:.4f} คะแนน")
        
        print(f"\n💡 สรุป:")
        if sum_real > sum_fake:
            print("   โมเดลตอบ 'จริง' เพราะผลรวมคะแนนฝั่งบวก ชนะ ฝั่งลบ")
            print("   (Feature Vector ตกอยู่ในพื้นที่ที่ Weight ของข่าวจริงทำงานได้ดีกว่า)")
        else:
            print("   โมเดลตอบ 'ปลอม' เพราะผลรวมคะแนนฝั่งลบ ชนะ ฝั่งบวก")

explain_decision_logic(0, data, model_gnn)


==================== 🧠 LOGIC EXPLAINER: Node 0 ====================
Final Real Score: 0.3001
Final Fake Score: -0.0462

✅ ปัจจัยบวก (ดันให้เป็นข่าวจริง):
   - Feature[212]: มีค่า 0.15 x Weight(0.26) = +0.0403 คะแนน
   - Feature[204]: มีค่า 0.17 x Weight(0.21) = +0.0368 คะแนน
   - Feature[207]: มีค่า 0.15 x Weight(0.25) = +0.0367 คะแนน

❌ ปัจจัยลบ (ดึงให้เป็นข่าวปลอม):
   - Feature[151]: มีค่า 0.11 x Weight(0.25) = +0.0283 คะแนน
   - Feature[85]: มีค่า 0.10 x Weight(0.28) = +0.0274 คะแนน
   - Feature[80]: มีค่า 0.11 x Weight(0.23) = +0.0262 คะแนน

💡 สรุป:
   โมเดลตอบ 'จริง' เพราะผลรวมคะแนนฝั่งบวก ชนะ ฝั่งลบ
   (Feature Vector ตกอยู่ในพื้นที่ที่ Weight ของข่าวจริงทำงานได้ดีกว่า)
